In [ ]:
%load ../lib/RecursiveSet.jsh
%load Formula.jsh

In [ ]:
@SafeVarargs
<T> RecursiveSet<T> set(T... elements) {
    return new RecursiveSet<>(elements);
}

The function `range` takes two natural numbers `start`and `stop`as arguments.  It
returns a set containing the numbers from `start` up to and including the number `stop`. 

In [ ]:
RecursiveSet<Integer> range(int start, int stop) {
    RecursiveSet<Integer> result = new RecursiveSet<>();
    for (int i = start; i <= stop; ++i) {
        result.add(i);
    }
    return result;
}

In [ ]:
range(1, 5)

# The [Zebra Puzzle](https://en.wikipedia.org/wiki/Zebra_Puzzle)

The following puzzle appeared in the magazine *Life International* on the 17th of December in the year 1962:
* There are five houses.
* The Englishman lives in the red house.
* The Spaniard owns the dog.
* Coffee is drunk in the green house.
* The Ukrainian drinks tea.
* The green house is immediately to the right of the ivory house.
* The Old Gold smoker owns snails.
* Kools are smoked in the yellow house.
* Milk is drunk in the middle house.
* The Norwegian lives in the first house.
* The man who smokes Chesterfields lives in the house next to the man with the fox.
* Kools are smoked in the house next to the house where the horse is kept.
* The Lucky Strike smoker drinks orange juice.
* The Japanese smokes Parliaments.
* The Norwegian lives next to the blue house.

Furthermore, each of the five houses is painted in a different colour, their inhabitants are of different nationalities, own different pets, drink different beverages, and smoke different brands of cigarettes.

Our task is to answers the following questions: 
1. *Who drinks water?*
2. *Who owns the zebra?*

## Choosing the Appropriate Variables

In order to solve this problem we use the following propositional variables, where the counter $i$ ranges from $1$ to $5$.  

* $\texttt{English}\langle i \rangle$ expresses the fact
  that the Englishman lives in house number $i$.
  
  The remaining nationalities are coded using the variables
  $\texttt{Spanish}\langle i \rangle$, $\texttt{Ukrainian}\langle i \rangle$,
  $\texttt{Norwegian}\langle i \rangle$, and $\texttt{Japanese}\langle i \rangle$.
* $\mathtt{Red}\langle i \rangle$ expresses that house number $i$ is red.
  
  We use the variables $\texttt{Green}\langle i \rangle$, $\texttt{Ivory}\langle i \rangle$,
  $\texttt{Yellow}\langle i \rangle$, and $\texttt{Blue}\langle i \rangle$ to encode the remaining
  colors.
* $\texttt{OldGold}\langle i \rangle$ expresses that the inhabitant of house number $i$ smokes cigarettes of the brand "Old Gold".
  
  We use the variables $\texttt{Kools}\langle i \rangle$,
  $\texttt{Chesterfields}\langle i \rangle$, $\texttt{Luckies}\langle i \rangle$, and
  $\texttt{Parliaments}\langle i \rangle$ to encode the cigarette brands.
* $\texttt{Dog}\langle i \rangle$ expresses the fact that the inhabitant of house number
  $i$ keeps a dog as his pet.

  We use the variables $\texttt{Snails}\langle i \rangle$, $\texttt{Fox}\langle i \rangle$,
  $\texttt{Horse}\langle i \rangle$, and $\texttt{Zebra}\langle i \rangle$ to encode the
  remaining pets.
* $\texttt{Coffee}\langle i \rangle$ expresses the fact the the inhabitant of house number $i$
  drinks coffee.

  We use the variables $\texttt{Milk}\langle i \rangle$,
  $\texttt{OrangeJuice}\langle i \rangle$, $\texttt{Tea}\langle i \rangle$, 
  and $\texttt{Water}\langle i \rangle$ to encode the remaining drinks.

We are using the angular brackets "$\langle$" and "$\rangle$" as part of the variable names 
because our parser for propositional logic accepts these symbols as part of variable names.  The parser would be confused if we would use parentheses.

## Loading the Necessary Files

Our goal is to solve this puzzle by first coding it as a solvability problem of propositional logic and then to solve the resulting set of clauses using the algorithm of Davis and Putnam.

In order to write formulas conveniently, we use the parser for propositional logic.

In [ ]:
%load PropositionalLogicParser.jsh

In order to be able to transform formulas from propositional logic into sets of clauses we load the file `04-CNF.jsh`, which implements the function `normalize` that takes a formula and transforms it into a set of clauses.

In [ ]:
%load 04-CNF.jsh

The file `06-Davis-Putnam.jsh` provides the functions `solve` and `complement`.

In [ ]:
%load 06-Davis-Putnam.jsh

In [ ]:
Literal neg(String p) {
    return new Neg(p);
}

Using the parser and the file `04-CNF.jsh` we can implement a function $\texttt{parseKNF}(s)$ that takes a string $s$ representing a formula and transforms $s$ into an equivalent set of clauses.

In [ ]:
RecursiveSet<RecursiveSet<Literal>> parseKNF(String s) {
    var f = parse(s);
    return normalize(f);
}

In [ ]:
parseKNF("a → b ↔ b → a")

## Auxiliary Functions

In order to succinctly express the constraints that all houses have different colours, the inhabitants have different nationalities etc., it is convenient to implement a function $\texttt{atMostOne}(V)$ that takes a set of variables $V$ and returns a set of formulas that is true if and only if all the variables from $V$ have different values.
$$ \texttt{atMostOne}(S) := 
   \bigl\{ \{\neg p, \neg q\} \bigm| \langle p, q \rangle \in S \times S \wedge
                                   p \not= q \bigr\}
$$

In [ ]:
RecursiveSet<RecursiveSet<Literal>> atMostOne(RecursiveSet<String> S) {
    return flatMap(S, p -> S.filterMap(q -> !p.equals(q),
                                       q -> set(neg(p), neg(q))));
}

Given a name $f$ and an index $i \in\{1,2,3,4,5\}$, the function 
$\texttt{varName}(i)$ creates the string 
$f\langle i \rangle$, e.g. the call `varName("Japanese", 2)` the following string:
```
Japanese<2>
```

In [ ]:
String varName(String f, int i) {
    return f + "<" + i + ">";
}

In [ ]:
varName("Japanese", 2)

A call of the form $\texttt{somewhere}(x)$ will return a clause that specifies that the person with property $x$ has to live in one of the houses from $1$ to $5$.  In order to be later able to insert this clause into a set of clauses, we have to make sure that we return a result of type `RecursiveSet<Literal>`.

In [ ]:
RecursiveSet<Literal> somewhere(String x) {
    return range(1, 5).map(i -> new Var(varName(x, i)));
}

In [ ]:
somewhere("Japanese")

Given an exclusive set of properties $S$ and a house number $i$, the function $\texttt{atMostOne}(S, i)$ returns a set of clauses that specifies that the person living in house number $i$ has at most one of the properties from the set $S$.  For example, if 
$S = \{\texttt{"Japanese"}, \texttt{"Englishman"}, \texttt{"Spaniard"}, \texttt{"Norwegian"}, \texttt{"Ukrainian"}\}$, 
then $\texttt{atMostOne}(S, 3)$ specifies that the inhabitant of house number 3 has at most one of the nationalities from the set $S$.

In [ ]:
RecursiveSet<RecursiveSet<Literal>> atMostOneAt(RecursiveSet<String> S, int i) {
    return atMostOne(S.map(x -> varName(x, i)));
}

In [ ]:
atMostOneAt(set("A", "B", "C"), 1)

The function $\texttt{onePerHouse}(S)$ is called as follows:
$$\texttt{onePerHouse}(\{\texttt{"Japanese"},
       \texttt{"English"}, 
       \texttt{"Spanish"}, \texttt{"Norwegian"}, 
       \texttt{"Ukrainian"}\})
$$
This function creates a set of clauses that expresses that there has to be a house where the Japanese lives, a house where the Englishman lives, a house where the Spaniard lives, a house where the Norwegian lives, and a house
where the Ukrainian lives.  Furthermore, the set of clauses would contain clauses that express that these five persons live in **different** houses.

In [ ]:
RecursiveSet<RecursiveSet<Literal>> onePerHouse(RecursiveSet<String> S) {
    RecursiveSet<RecursiveSet<Literal>> Result = S.map(x -> somewhere(x));
    return Result.union(flatMap(range(1, 5), i -> atMostOneAt(S, i)));
}

In [ ]:
onePerHouse(set("A", "B"))

Given two properties $a$ and $b$ the function $\texttt{sameHouse}(a, b)$ computes a set of clauses that specifies that if the inhabitant of house number $i$ has the property $a$, then he also has the the property $b$ and vice versa.  For example, $\texttt{sameHouse}(\texttt{"Japanese"}, \texttt{"Dog"})$ specifies that the Japanese guy keeps a dog.

In [ ]:
RecursiveSet<RecursiveSet<Literal>> sameHouse(String a, String b) {
    return flatMap(range(1, 5),
                   i -> parseKNF(varName(a, i) + " ↔ " + varName(b, i)));
}

In [ ]:
sameHouse("Red", "Tea")

Given to properties $a$ and $b$ the function $\texttt{nextTo}(a, b)$ computes a set of clauses that specifies that the inhabitants with properties $a$ and $b$ are direct neighbours.  For example, $\texttt{nextTo}(\texttt{'Japanese'}, \texttt{'Dog'})$ specifies that the Japanese guy lives next to the guy who keeps a dog.

In [ ]:
RecursiveSet<RecursiveSet<Literal>> nextTo(String a, String b) {
    List<String> Formulas = new ArrayList<>();
    Formulas.add(varName(a, 1) + " → " + varName(b, 2));
    for (int i = 2; i <= 4; ++i) {
        Formulas.add(varName(a, i) + " → " + varName(b, i - 1) + " ∨ " + varName(b, i + 1));
    }
    Formulas.add(varName(a, 5) + " → " + varName(b, 4));
    return flatMap(Formulas, formula -> parseKNF(formula));
}

In [ ]:
nextTo("A", "B")

Given two properties $a$ and $b$ the function $\texttt{leftTo}(a, b)$ computes a set of clauses that specifies that the inhabitant with property $a$ lives in the house to the left of the inhabitant who has property $b$.  For example, $\texttt{leftTo}(\texttt{"Japanese"}, \texttt{"Dog"})$ specifies that the Japanese guy lives in the house to the left of the house where there is a dog.

In [ ]:
RecursiveSet<RecursiveSet<Literal>> leftTo(String a, String b) {
    List<String> Formulas = new ArrayList<>();
    for (int i = 1; i <= 4; ++i) {
        Formulas.add(varName(a, i) + " → " + varName(b, i + 1));
    }
    Formulas.add("¬" + varName(a, 5));
    return flatMap(Formulas, formula -> parseKNF(formula));
}

In [ ]:
leftTo("A", "B")

In [ ]:
var Nations = set("English", "Spanish", "Ukrainian", "Norwegian", "Japanese");
var Drinks  = set("Coffee", "Tea", "Milk", "OrangeJuice", "Water");
var Pets    = set("Dog", "Snails", "Horse", "Fox", "Zebra");
var Brands  = set("Luckies", "Parliaments", "Kools", "Chesterfields", "OldGold");
var Colours = set("Red", "Green", "Ivory", "Yellow", "Blue");

The function `allClauses` returns a set of clauses describing the problem.

In [ ]:
RecursiveSet<RecursiveSet<Literal>> allClauses() {
    // 1. Base constraints: Every property category gets exactly one house
    var Result = flatMap(List.of(Nations, Drinks, Pets, Brands, Colours), S -> onePerHouse(S));
    // 2. The specific puzzle clues
    var puzzleClues = List.of(
        sameHouse("English", "Red"),         // The Englishman lives in the red house.
        sameHouse("Spanish", "Dog"),         // The Spaniard owns the dog.
        sameHouse("Coffee", "Green"),        // Coffee is drunk in the green house.
        sameHouse("Ukrainian", "Tea"),       // The Ukrainian drinks tea.
        leftTo("Ivory", "Green"),            // The green house is immediately to
                                             // the right of the ivory house.
        sameHouse("OldGold", "Snails"),      // The Old Gold smoker owns snails.
        sameHouse("Kools", "Yellow"),        // Kools are smoked in the yellow house.
        parseKNF(varName("Milk", 3)),        // Milk is drunk in the middle house.
        parseKNF(varName("Norwegian", 1)),   // The Norwegian lives in the first house.
        nextTo("Chesterfields", "Fox"),      // The Chesterfields smoker lives next to
                                             // the fox owner.
        nextTo("Kools", "Horse"),            // Kools are smoked next to the horse.
        sameHouse("Luckies", "OrangeJuice"), // The Lucky Strike smoker drinks orange juice.
        sameHouse("Japanese", "Parliaments"),// The Japanese smokes Parliaments.
        nextTo("Norwegian", "Blue")          // The Norwegian lives next to the blue house.
    );
    return Result.union(flatMap(puzzleClues, S -> S));
}

In [ ]:
var Clauses = allClauses();
for (var clause : Clauses) {
    System.out.println(clause);
}

In [ ]:
Clauses.size()

In [ ]:
RecursiveSet<RecursiveSet<Literal>> main() {
    return solve(allClauses());
}

Solving the problem takes less than a second.

In [ ]:
long start = System.currentTimeMillis();
var Solution = main();
System.out.println("Solution: " + (System.currentTimeMillis() - start) + " ms");

In [ ]:
Solution

## Functions to PrettyPrint the Solution

We use the function `arb`, which returns a random element of a set.  This function is defined in the file `06-Davis-Putnam.jsh`.  Note that we must not define it again here:  In JShell, redefining a method resets all variables whose values have been computed using this method.  Since the variable `Solution` has been computed by a call of `solve`, which calls `arb`, redefining `arb` would reset `Solution` to `null`.

The function `extractAssignment` takes a solution and returns a map that maps every property to the number of the house of the inhabitant that has this property.  The variable names have the form `name<i>`, where `i` is a single digit.

In [ ]:
Map<String, Integer> extractAssignment(RecursiveSet<RecursiveSet<Literal>> Solution) {
    Map<String, Integer> Assignment = new TreeMap<>();
    for (var Unit : Solution) {
        if (arb(Unit) instanceof Var(var lit)) {
            int    number = Integer.parseInt(lit.substring(lit.length() - 2, lit.length() - 1));
            String name   = lit.substring(0, lit.length() - 3);
            Assignment.put(name, number);
        }
    }
    return Assignment;
}

In [ ]:
extractAssignment(Solution)

The function `display`, which is provided by the IJava kernel, can be used to present the solution graphically.

In [ ]:
void showHTML(RecursiveSet<RecursiveSet<Literal>> Solution) {
    var Assignment = extractAssignment(Solution);
    var result     = new StringBuilder();
    result.append("<table style=\"border:2px solid blue\">\n");
    result.append("<tr>");
    for (var name : List.of("House", "Nationality", "Drink", "Animal", "Brand", "Colour")) {
        result.append("<th style=\"color:gold; background-color:blue\">" + name + "</th>");
    }
    result.append("</tr>\n");
    for (int house = 1; house <= 5; house++) {
        result.append("<tr><td style=\"border:1px solid green\">" + house + "</td>");
        for (var Class : List.of(Nations, Drinks, Pets, Brands, Colours)) {
            for (var x : Class) {
                if (Assignment.get(x) == house) {
                    result.append("<td style=\"border:1px solid green\">" + x + "</td>");
                }
            }
        }
        result.append("</tr>\n");
    }
    result.append("</table>");
    display(result.toString(), "text/html");
}

In [ ]:
showHTML(Solution);

## Checking the Uniqueness of the Solution

Given a set of unit clauses `UnitClauses`, the function `negateSolution(UnitClauses)` returns a clause that is the logical negation of `UnitClauses`.

In [ ]:
RecursiveSet<Literal> negateSolution(RecursiveSet<RecursiveSet<Literal>> UnitClauses) {
    return UnitClauses.map(unit -> complement(arb(unit)));
}

In [ ]:
RecursiveSet<RecursiveSet<Literal>> input = set();
RecursiveSet<Literal> c1 = set(new Var("a"));
RecursiveSet<Literal> c2 = set(neg("b"));
input.add(c1);
input.add(c2);
negateSolution(input)

In [ ]:
void checkUniqueness(RecursiveSet<RecursiveSet<Literal>> Solution,
                     RecursiveSet<RecursiveSet<Literal>> Clauses) {
    var negationLiterals = negateSolution(Solution);
    var alternative      = solve(Clauses.union(set(negationLiterals)));
    RecursiveSet<Literal> EmptyClause = set();
    if (alternative.contains(EmptyClause)) {
        System.out.println("The solution is unique!");
    } else {
        System.out.println("ERROR: The solution is not unique!");
    }
}

In [ ]:
long start = System.currentTimeMillis();
checkUniqueness(Solution, Clauses);
System.out.println("uniqueness: " + (System.currentTimeMillis() - start) + " ms");